# Single-tool eval: `query_loki`

Agent loop built from scratch (no `agent.agent` import), bound to `query_loki` **plus** its metric/label discovery helpers (`get_loki_label_values`) from `agent.tools.loki`. Everything outside this backend is still unavailable. This measures how far the model gets on this backend alone.


In [ ]:
from eval_transcript import REFUSAL, run_eval_repeats
from agent.tools.loki import get_loki_label_values, query_loki

# Runs per question. One is enough at temperature 0: the loop is
# deterministic, so repeats reproduce the same answer and the same cost.
RUNS = 1


### System prompt

Deliberately **not** a restatement of the tool list. Each tool's own docstring is already sent to the model as part of the bound schema, so repeating it here only creates a second copy to drift out of sync — an earlier version of this prompt duplicated `get_loki_label_values`'s docstring verbatim, wrong label examples and all. What stays here is policy the schema can't carry: how to answer, what to disclose, and what not to do.


In [ ]:
SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
Today is 2026-08-25. The Loki backend is read-only and holds roughly the last week of logs.

Tool use:
- Never guess a Loki label name or value. Call `get_loki_label_values` to confirm one exists before you use it in a LogQL selector.
- If a tool call fails, do not repeat the identical call. Try a different approach or say you cannot answer.

Answering:
- Say how many log lines you actually looked at, out of how many matched, and the time range they cover. If you saw only a sample, say so before drawing any conclusion from it.
- Rank causes by how much evidence supports them. Do not lead with a one-off line.
- At most 5 sentences. Plain prose: no markdown headings, no bold, no numbered sections.
- You are read-only. Recommend actions for a human operator; never imply you can restart or change anything.
- If these tools cannot answer the question, say so plainly instead of guessing.
"""

TOOLS = [get_loki_label_values, query_loki]

### The loop

Every run below goes through `eval_transcript.run_eval_loop`, shared across all the eval notebooks: it sends the messages, dispatches tool calls, prints each call's token counts and wall-clock, and — on every run — dumps the full transcript (every message sent, every response, every tool call/result, per-call token usage, the grade) to `eval/logs/<timestamp>__<label>.json`.

`run_eval_repeats` runs the same question three times. Temperature 0 is not a determinism guarantee on any of these providers, and a finding that shows up in one run of three is noise.


### Is the live backend up?

Both live questions below need a reachable Loki. If it isn't reachable, the runs measure error handling and nothing else — that is not a result, so skip them rather than recording a void transcript as if it were one.


In [ ]:
try:
    await get_loki_label_values.ainvoke({"label": "job"})
    LIVE_UP = True
except Exception as exc:
    LIVE_UP = False
    print(
        f"Loki unreachable ({exc}) — skipping the live cells, running the eval-dataset backend only."
    )

## In-scope question

A question the bound tools (`get_loki_label_values`, `query_loki`) together should be able to answer.

**Look for:** does it confirm the real label names before selecting on them, and does it report how much it saw?


In [ ]:
if LIVE_UP:
    await run_eval_repeats(
    "06_query_loki:live-inscope",
    TOOLS,
    SYSTEM_PROMPT,
    "Show recent error logs across all services.",
    expect_tools=["query_loki", "get_loki_label_values"],
    runs=RUNS,
)


## Out-of-scope question

`query_loki` is instant-only; `query_loki_range` exists but is deliberately **not** bound here, so no bound tool can scope a 15-minute window.

**Look for:** does the model say it can't scope by time, or does it quietly answer as if it could? A refusal that blames a connection error doesn't count — that's the backend being down, not the model recognising the limit, which is why the reachability guard above matters.


In [ ]:
if LIVE_UP:
    await run_eval_repeats(
    "06_query_loki:live-outofscope",
    TOOLS,
    SYSTEM_PROMPT,
    "Show error logs from the last 15 minutes.",
    expect_tools=["query_loki"],
    runs=RUNS,
)


### Ground truth

Computed from the fixture, for `machine='lambda2'`, so the answer can be graded instead of admired:

|                                   | over all error lines                           | in the newest 20 (what the tool returns)       |
| --------------------------------- | ---------------------------------------------- | ---------------------------------------------- |
| lines at `detected_level='error'` | **1,246**                                      | 20                                             |
| containers                        | loki 1,553, **node-exporter 131**, promtail 19 | loki 19, node-exporter 1                       |
| dominant signature                | `ratestore.go … unable to get store` (**504**) | `scheduler_processor.go` (**16**), ratestore 3 |
| span                              | 2026-08-20 11:30 → 2026-08-22 17:53            | 2026-08-22 14:00 → 17:53                       |

Note the two columns disagree, and that is the point: ranking causes over the whole 1,246 gives a different answer than ranking over the newest 20. So a good answer must **say which population it is describing** — hence the `expect` check on the total, `1246`, below, and on `node-exporter`, the evidence class the old first-N-in-file window hid entirely.

`expect` is a crude substring check; it only catches an answer that omits a thing outright, which is exactly the sampling blind spot worth automating. Whether what it _did_ say is correct is still a human read of the transcript. (Worth knowing what it cannot judge: an early version of this check demanded the string `ratestore`, and failed a correct answer for not quoting a Go filename it had no reason to quote.)


### Ground truth

Computed from the fixture, for `machine='lambda2'`, so the answer can be graded instead of admired:

|                                   |                                                                                                          |
| --------------------------------- | -------------------------------------------------------------------------------------------------------- |
| lines at `detected_level='error'` | **1,246**                                                                                                |
| containers, by volume             | **loki 1,553**, **node-exporter 131**, promtail 19 _(lines containing "error")_                          |
| dominant signatures               | `ratestore.go … unable to get store` (**504**), `scheduler_processor.go` query-frontend errors (**478**) |
| `err="empty ring"`                | **1** — transient, and not a headline                                                                    |
| span of error lines               | 2026-08-20 11:30 → 2026-08-22 17:53                                                                      |

So a good answer leads with the ratestore/scheduler volume, mentions node-exporter, and discloses that it saw a sample. `expect` below is a crude substring check — it only catches an answer that omits a thing entirely, which is exactly the sampling blind spot worth automating. Whether what it _did_ say is correct is still a human read of the transcript.


In [ ]:
from agent.tools.helper import query_eval_logs

EVAL_TOOLS = [query_eval_logs]
EVAL_SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
Today is 2026-08-25. You are reading a fixed log dataset covering 2026-08-20 to 2026-08-24 for three machines (marconi-01, marconi-02, lambda2). It is a snapshot, not live infrastructure.

Answering:
- Read the JSON records the tool returns and answer in plain English. Never paste raw JSON back at the user.
- Say how many log lines you actually looked at, out of how many matched, and the time range they cover. If you saw only a sample, say so before drawing any conclusion from it.
- Rank causes by how much evidence supports them. Do not lead with a one-off line.
- At most 5 sentences. Plain prose: no markdown headings, no bold, no numbered sections.
- You are read-only. Recommend actions for a human operator; never imply you can restart or change anything.
- If the tool cannot answer the question, say so plainly instead of guessing.
"""

await run_eval_repeats(
    "06_query_loki:eval",
    EVAL_TOOLS,
    EVAL_SYSTEM_PROMPT,
    "What errors are showing up in the lambda2 logs, and what's likely going on?",
    expect=["node-exporter", "1246"],
    expect_tools=["query_eval_logs"],
    runs=RUNS,
)
